# Collaborative Filtering item-based (ItemKNN / co-click) — con fallback di popolarità

Questo notebook implementa un ItemKNN costruito sui segnali di co-click del TRAIN, integrato con un piccolo fallback di popolarità per gestire i casi di cold-start / copertura nulla. È più efficace in pratica, ma può essere considerato un hybrid leggero (CF + popolarità).

Dataset: MINDsmall (`behaviors.tsv`).

Metriche sul DEV (a livello di impression): AUC, MRR, nDCG@5, nDCG@10, P@5, R@5.
Tie-breaking: deterministico ogni volta che è richiesto un ranking rigoroso.

In [ ]:
# --- 1) Import e caricamento dei dati ---
# Carichiamo i comportamenti TRAIN per costruire le similarità ItemKNN sui co-click.
# Valutiamo sui comportamenti DEV (nessun leakage).
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from math import sqrt
from tqdm import tqdm

from sklearn.metrics import roc_auc_score

behav_cols = ['imp_id', 'user_id', 'time', 'history', 'impressions']
train_behav = pd.read_csv('../dataset/MINDsmall_train/behaviors.tsv', sep='\t', names=behav_cols)
dev_behav   = pd.read_csv('../dataset/MINDsmall_dev/behaviors.tsv',   sep='\t', names=behav_cols)

print('Impression di training:', len(train_behav))
print('Impression di dev:', len(dev_behav))
print('Utenti unici nel training:', train_behav['user_id'].nunique())


## Split dei dati e controllo del leakage

- I dati TRAIN vengono usati per costruire il modello di similarità item-item.
- I dati DEV vengono usati esclusivamente per la valutazione.

Questo evita ogni forma di data leakage: nessuna statistica viene calcolata a partire dal DEV.

In [ ]:
# --- 2) Funzioni di parsing ---
def parse_history(hist_str: str):
    """Restituisce la lista di news_id nel campo history."""
    if pd.isna(hist_str):
        return []
    s = str(hist_str).strip()
    return s.split() if s else []

def parse_clicked_from_impressions(imp_str: str):
    """Restituisce i news_id cliccati a partire da una stringa di impression con etichette."""
    if pd.isna(imp_str):
        return []
    items = str(imp_str).split()
    return [it.split('-')[0] for it in items if it.endswith('-1')]

def parse_candidates_with_labels(imp_str: str):
    """Le impression del DEV hanno la forma 'Nxxx-0 Nyyy-1 ...'."""
    items = str(imp_str).split()
    y_true = np.array([int(it.split('-')[1]) for it in items], dtype=int)
    cand = [it.split('-')[0] for it in items]
    return cand, y_true


## Costruzione dei co-click locali

Costruiamo i conteggi di co-occorrenza usando una **finestra locale**: per ogni riga del TRAIN, prendiamo gli ultimi `WINDOW` articoli della cronologia utente più gli articoli cliccati nell'impression corrente.
Questo riduce le co-occorrenze rumorose a lungo raggio, molto comuni nella raccomandazione di news.

In [ ]:
# --- 3) Conteggio locale dei co-click (TRAIN) ---
# WINDOW: quanti articoli recenti della cronologia includere per impression (contesto simile a una sessione)
# MAX_HIST: limite di sicurezza per evitare cronologie molto lunghe
# MIN_CO: ignora le co-occorrenze singole (spesso rumore)
WINDOW = 10
MAX_HIST = 50
MIN_CO = 2

item_cnt = Counter()               # item_cnt[i] = occorrenze nelle finestre locali
co_cnt = defaultdict(Counter)      # co_cnt[i][j] = conteggio di co-occorrenza

for _, row in tqdm(train_behav.iterrows(), total=len(train_behav), desc='Conteggio co-click (locale)'):
    hist = parse_history(row['history'])
    clk  = parse_clicked_from_impressions(row['impressions'])

    if len(hist) > MAX_HIST:
        hist = hist[-MAX_HIST:]

    items = hist[-WINDOW:] + clk
    items = list(dict.fromkeys(items))  # rimuove i duplicati, mantiene l'ordine
    if len(items) == 0:
        continue

    for x in items:
        item_cnt[x] += 1

    for i in range(len(items)):
        a = items[i]
        for j in range(i+1, len(items)):
            b = items[j]
            co_cnt[a][b] += 1
            co_cnt[b][a] += 1

print('Articoli unici con interazioni:', len(item_cnt))


## Similarità item-item (ItemKNN)

Calcoliamo una similarità in stile coseno con shrinkage:

`sim(i,j) = c_ij / (sqrt(c_i * c_j) + SHRINK)`

Lo shrinkage penalizza le co-occorrenze deboli e stabilizza le similarità.
Manteniamo solo i top-`TOPK_NEIGHBORS` vicini per ciascun articolo.

In [ ]:
# --- 4) Cosine similarity + shrinkage + vicini top-K ---
TOPK_NEIGHBORS = 100
SHRINK = 10.0

neighbors = {}  # neighbors[i] = lista di (j, sim), ordinata per sim decrescente

for i, nbrs in tqdm(co_cnt.items(), total=len(co_cnt), desc='Costruzione ItemKNN'):
    ci = item_cnt[i]
    sims = []
    for j, cij in nbrs.items():
        if cij < MIN_CO:
            continue
        cj = item_cnt[j]
        sim = cij / (sqrt(ci * cj) + SHRINK)
        sims.append((j, sim))
    sims.sort(key=lambda x: x[1], reverse=True)
    if sims:
        neighbors[i] = sims[:TOPK_NEIGHBORS]

print('Articoli con lista di vicini:', len(neighbors))


## Scoring dei candidati

Per ogni impression, assegniamo un punteggio a ciascun articolo candidato sommando le similarità con gli articoli nella cronologia recente dell'utente:

`score(c) = Σ_{h in history} sim(h, c)`

Lo scoring è implementato in modo efficiente, iterando sulle liste di vicini degli articoli in cronologia e accumulando i punteggi solo per i candidati presenti nell'impression corrente.

In [ ]:
# --- 5) Scoring ItemKNN + fallback di popolarità ---
# NOTA: in pratica più efficace, può essere considerato un hybrid leggero (CF + popolarità).
MAX_HISTORY_FOR_SCORING = 30
ALPHA = 0.7  # peso della componente CF

def get_popularity_scores(behav_df: pd.DataFrame) -> Counter:
    """Popolarità globale dai click del TRAIN: conta gli articoli con etichetta '-1'."""
    clicked = behav_df['impressions'].dropna().astype(str).str.split().explode()
    clicked = clicked[clicked.str.endswith('-1')].str.split('-').str[0]
    return Counter(clicked)

pop_scores = get_popularity_scores(train_behav)
pop_max = max(pop_scores.values()) if len(pop_scores) else 1

pop_total = float(sum(pop_scores.values())) if len(pop_scores) else 1.0

def item_novelty(news_id: str) -> float:
    """
    Novelty = -log2(p(item)), con p calcolata dalla distribuzione dei click nel TRAIN.
    Per gli articoli mai visti si usa count=1 (popolarità minima plausibile) invece di eps,
    per evitare una novelty artificialmente infinita.
    """
    c = pop_scores.get(news_id, 1)  # <-- punto chiave: articolo mai visto -> 1 click
    p = c / pop_total
    return float(-np.log2(p))

def pop_norm(news_id: str) -> float:
    """Popolarità log-normalizzata in [0,1]."""
    return np.log1p(pop_scores.get(news_id, 0)) / np.log1p(pop_max)

def score_candidates_itemknn(history_list, candidate_ids, neighbors_dict):
    """Scoring ibrido: alpha*ItemKNN + (1-alpha)*popolarità."""
    if len(history_list) > MAX_HISTORY_FOR_SCORING:
        history_list = history_list[-MAX_HISTORY_FOR_SCORING:]

    cand_set = set(candidate_ids)
    score_map = defaultdict(float)

    for h in history_list:
        if h not in neighbors_dict:
            continue
        for j, sim in neighbors_dict[h]:
            if j in cand_set:
                score_map[j] += sim

    cf = np.array([score_map.get(c, 0.0) for c in candidate_ids], dtype=float)
    pop = np.array([pop_norm(c) for c in candidate_ids], dtype=float)
    return ALPHA * cf + (1 - ALPHA) * pop


## Valutazione offline sul DEV

Valutiamo le prestazioni di ranking a livello di impression usando:
AUC (quando entrambe le classi sono presenti), MRR (primo rilevante), nDCG@5, nDCG@10, Precision@5, Recall@5.

La valutazione è deterministica (nessuna componente casuale).

In [ ]:
def deterministic_order(y_score: np.ndarray, cand_ids: list[str]) -> np.ndarray:
    """
    Restituisce gli indici ordinati in modo deterministico per (-score, cand_id).
    """
    y_score = np.asarray(y_score, dtype=np.float64)
    cand_ids = np.asarray(cand_ids, dtype=str)
    # np.lexsort ordina prima per l'ultima chiave -> vogliamo chiave primaria = -score, secondaria = cand_id
    return np.lexsort((cand_ids, -y_score))

In [ ]:
# --- 6) Metriche e ciclo di valutazione ---
def mrr_first_hit(y_true: np.ndarray, y_score: np.ndarray, cand_ids: list[str]) -> float:
    """MRR standard: reciproco del rank del primo elemento rilevante (parità deterministiche)."""
    order = deterministic_order(y_score, cand_ids)
    y_sorted = y_true[order]
    hits = np.where(y_sorted == 1)[0]
    return 1.0 / (hits[0] + 1) if len(hits) > 0 else 0.0

def ndcg_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    """Calcola nDCG@k con un'implementazione locale."""
    def dcg_at_k_local(y_true_local, y_score_local, k_local):
        y_true_local = np.asarray(y_true_local)
        y_score_local = np.asarray(y_score_local)
        order = np.argsort(-y_score_local, kind="mergesort")[:k_local]
        gains = y_true_local[order]
        discounts = np.log2(np.arange(2, len(gains) + 2))
        return float(np.sum(gains / discounts))

    def ideal_dcg_at_k_local(y_true_local, k_local):
        y_true_local = np.asarray(y_true_local)
        ideal = np.sort(y_true_local)[::-1][:k_local]
        discounts = np.log2(np.arange(2, len(ideal) + 2))
        return float(np.sum(ideal / discounts))

    idcg = ideal_dcg_at_k_local(y_true, k)
    if idcg == 0.0:
        return 0.0
    return dcg_at_k_local(y_true, y_score, k) / idcg

def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, cand_ids: list[str], k: int = 5) -> float:
    order = deterministic_order(y_score, cand_ids)[:k]
    return float(np.mean(y_true[order])) if len(order) else 0.0

def recall_at_k(y_true: np.ndarray, y_score: np.ndarray, cand_ids: list[str], k: int = 5) -> float:
    order = deterministic_order(y_score, cand_ids)[:k]
    denom = int(np.sum(y_true))
    return float(np.sum(y_true[order]) / denom) if denom > 0 else 0.0

def novelty_at_k(y_score: np.ndarray, cand_ids: list[str], k: int = 5) -> float:
    order = deterministic_order(y_score, cand_ids)[:k]
    if len(order) == 0:
        return 0.0
    top_ids = [cand_ids[i] for i in order]
    return float(np.mean([item_novelty(nid) for nid in top_ids]))

metrics = {'auc': [], 'mrr': [], 'ndcg@5': [], 'ndcg@10': [], 'p@5': [], 'r@5': [], 'novelty@5': [], 'novelty@10': []}

print('Valutazione sul DEV...')
for _, row in tqdm(dev_behav.iterrows(), total=len(dev_behav), desc='DEV'):
    hist = parse_history(row['history'])
    candidates, y_true = parse_candidates_with_labels(row['impressions'])
    y_score = score_candidates_itemknn(hist, candidates, neighbors)

    n_pos = int(y_true.sum())
    if 0 < n_pos < len(y_true):
        metrics['auc'].append(roc_auc_score(y_true, y_score))

    if n_pos > 0:
        metrics['mrr'].append(mrr_first_hit(y_true, y_score, candidates))
        metrics['ndcg@5'].append(ndcg_at_k(y_true, y_score, k=5))
        metrics['ndcg@10'].append(ndcg_at_k(y_true, y_score, k=10))
        metrics['p@5'].append(precision_at_k(y_true, y_score, candidates, k=5))
        metrics['r@5'].append(recall_at_k(y_true, y_score, candidates, k=5))
        metrics['novelty@5'].append(novelty_at_k(y_score, candidates, k=5))
        metrics['novelty@10'].append(novelty_at_k(y_score, candidates, k=10))

results = {k: float(np.mean(v)) if len(v) else float('nan') for k, v in metrics.items()}

print('\n' + '='*42)
print('Collaborative Filtering item-based (ItemKNN / co-click) — con fallback di popolarità — prestazioni DEV')
print('='*42)
for k, v in results.items():
    print(f"{k.upper():<7}: {v:.4f}")


In [ ]:
print("\nRiepilogo (DEV):")
print(
    f"AUC={results['auc']:.4f}, MRR={results['mrr']:.4f}, "
    f"nDCG@5={results['ndcg@5']:.4f}, nDCG@10={results['ndcg@10']:.4f}, "
    f"P@5={results['p@5']:.4f}, R@5={results['r@5']:.4f}, "
    f"Novelty@5={results['novelty@5']:.4f}, Novelty@10={results['novelty@10']:.4f}"
)

## Opzionale: output di ranking deterministico

Se è richiesta una lista di ranking rigorosa, le parità vengono risolte in modo deterministico tramite:
(score decrescente, news_id crescente).
Questo garantisce file di output riproducibili.

In [ ]:
# --- 7) Output di ranking deterministico (opzionale) ---
def generate_itemknn_ranks(behav_df: pd.DataFrame, neighbors_dict) -> list[str]:
    lines = []
    for _, row in behav_df.iterrows():
        imp_id = row['imp_id']
        hist = parse_history(row['history'])
        raw = str(row['impressions']).split()
        candidates = [c.split('-')[0] for c in raw]
        scores = score_candidates_itemknn(hist, candidates, neighbors_dict)

        order = sorted(range(len(candidates)), key=lambda i: (-scores[i], candidates[i]))
        ranks = [0] * len(candidates)
        for r, idx in enumerate(order, start=1):
            ranks[idx] = r

        lines.append(f"{imp_id} [{' '.join(map(str, ranks))}]")
    return lines

# Decommentare per generare il file
# output_lines = generate_itemknn_ranks(dev_behav, neighbors)
# with open('itemknn_prediction.txt', 'w') as f:
#     f.write('\n'.join(output_lines))
# print('itemknn_prediction.txt salvato')

## Conclusioni — ItemKNN + fallback di popolarità (hybrid leggero)

### Riepilogo del metodo
Abbiamo esteso il modello ItemKNN puro basato su co-click con un **fallback di popolarità** derivato dai conteggi di click sul TRAIN.
Il punteggio finale del candidato è una combinazione pesata:
$$score(c) = \alpha \cdot score_{CF}(c) + (1-\alpha) \cdot score_{pop}(c)$$
Questo migliora la robustezza quando la componente CF ha bassa copertura (es. cronologia sparsa, candidati mai visti).

### Risultati sul DEV
- **AUC:** 0.5318
- **MRR:** 0.2746
- **nDCG@5:** 0.2465
- **nDCG@10:** 0.3102
- **Precision@5:** 0.1063
- **Recall@5:** 0.3634
- **Novelty@5:** 13.1188
- **Novelty@10:** 14.1046

### Interpretazione
L'aggiunta della popolarità produce un **miglioramento costante su tutte le metriche orientate all'accuratezza** (AUC/MRR/nDCG/P@5/R@5) rispetto alla baseline ItemKNN pura. Questo indica che, su MIND-small, una larga parte delle impression soffre di **sparsità del segnale CF**, e un prior globale (la popolarità) aiuta a risolvere le parità e fornisce un ranking significativo quando il grafo di co-click non collega la cronologia dell'utente ai candidati.
Come atteso, il guadagno in accuratezza comporta una diminuzione della **Novelty@K**, riflettendo il classico **trade-off tra accuratezza e novelty**.

### Limiti e implicazioni
- Questo approccio è di fatto un **hybrid leggero** (CF + popolarità), non un metodo di "CF puro".
- La popolarità può aumentare l'exposure bias e ridurre diversità/novelty.
- **Indicazione per l'hybrid finale**: il fallback di popolarità è uno stabilizzatore efficace e può essere combinato con gli score content-based (es. embedding testuali) per migliorare sia la copertura sia la personalizzazione, controllando novelty/diversità tramite re-ranking o pesatura.